In [ ]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMNet, GRUNet
from algorithm import fedlbs

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')

else:
    device = torch.device("cpu")

running on gpu


# Parameters

In [ ]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 256
n_layers = 2
lr = 0.001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training
num_clients_per_round = 5 #number of clients to participate in training

# Data Preperation

In [ ]:
dataframes = fetch_dataset()

Floor3
Floor4
Floor5
Floor6
Floor7
Floor3
Floor4
Floor5
Floor6
Floor7


In [ ]:
n_clients = len(dataframes) #total number of clients to partition data into

In [ ]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', 'z4_Plug(kW)', 'z5_AC1(kW)', 'z5_Light(kW)',
       'z5_Plug(kW)', 'Year', 'Month', 'Day', 'Hour', 'Minute'],
      dtype='object')
(79056, 23)
Index(['total_demand', 'z1_AC1(kW)', 'z1_AC2(kW)', 'z1_AC3(kW)', 'z1_AC4(kW)',
       'z1_Light(kW)', 'z1_Plug(kW)', 'z2_AC1(kW)', 'z2_Light(kW)',
       'z2_Plug(kW)', 'z3_Light(kW)', 'z3_Plug(kW)', 'z4_AC1(kW)',
       'z4_Light(kW)', '

# Build the training set

In [ ]:
train_loader = []
test_loader = []
label_scalers = []

for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features

    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )

    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler()

    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))

    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])

    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)

    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to('cpu'), torch.from_numpy(train_y).to('cpu'))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to('cpu'), torch.from_numpy(test_y).to('cpu'))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch

    # release some memory
    del train_x, train_y

input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(78966, 90, 23) (78966, 1)
(68823, 90, 23) (68823, 1)
(78966, 90, 23) (78966, 1)


# LSTM model

In [ ]:
lstm = LSTMNet(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

LSTMNet(
  (lstm): LSTM(23, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (relu): ReLU()
)
814337


## S =3

In [ ]:
%%time
S = 3
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [2 0 1 3 4]
round 0, starting client 1/5, id: 2
Epoch [1/1], Train Loss: 0.013412045559308635
round 0, starting client 2/5, id: 0
Epoch [1/1], Train Loss: 0.015722869281951896
round 0, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.015089800894321477
round 0, starting client 4/5, id: 3
Epoch [1/1], Train Loss: 0.017645187131217075
round 0, starting client 5/5, id: 4
Epoch [1/1], Train Loss: 0.027573652882281627

sMAPE: 18.7512222259162%
calc smape: 36.97024630484328%
MAE: 8.642097602710027
RMSE: 15.25666077837374
Average Loss:  0.010767613298343724

starting average round 1
clients:  [2 0 1 3 4]
round 1, starting client 1/5, id: 2
Epoch [1/1], Train Loss: 0.0049021067721607015
round 1, starting client 2/5, id: 0
Epoch [1/1], Train Loss: 0.006309305400627933
round 1, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.006213963877211219
round 1, starting client 4/5, id: 3
Epoch [1/1], Train Loss: 0.007391700253136596
round 1, starting client 

In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}.npy', rmse_lbs)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning:

Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.



## S = 5

In [ ]:
%%time
S = 5
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [2 0 1 3 4]
round 0, starting client 1/5, id: 2
Epoch [1/1], Train Loss: 0.014556295590543908
round 0, starting client 2/5, id: 0
Epoch [1/1], Train Loss: 0.016136238590020252
round 0, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.015133526861347012
round 0, starting client 4/5, id: 3
Epoch [1/1], Train Loss: 0.016337520003599944
round 0, starting client 5/5, id: 4
Epoch [1/1], Train Loss: 0.022961588889421493

sMAPE: 62.40478650509796%
calc smape: 41.222159870126546%
MAE: 7.82725272179913
RMSE: 14.619418692479709
Average Loss:  0.00981893011705355

starting average round 1
clients:  [2 0 1 3 4]
round 1, starting client 1/5, id: 2
Epoch [1/1], Train Loss: 0.004862414900791742
round 1, starting client 2/5, id: 0
Epoch [1/1], Train Loss: 0.006928317630123186
round 1, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.0063402633264767715
round 1, starting client 4/5, id: 3
Epoch [1/1], Train Loss: 0.006988316581074925
round 1, starting client

In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}.npy', rmse_lbs)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning:

Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.



## S= 7

In [ ]:
%%time
S = 7
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [0 2 1 4 3]
round 0, starting client 1/5, id: 0
Epoch [1/1], Train Loss: 0.01474039792464894
round 0, starting client 2/5, id: 2
Epoch [1/1], Train Loss: 0.016076027818375315
round 0, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.015218214885728651
round 0, starting client 4/5, id: 4
Epoch [1/1], Train Loss: 0.016185144111865536
round 0, starting client 5/5, id: 3
Epoch [1/1], Train Loss: 0.023014819072025103

sMAPE: 99.62256597639605%
calc smape: 41.42623599603126%
MAE: 7.7781518356605925
RMSE: 14.691487513957608
Average Loss:  0.009904515297295791

starting average round 1
clients:  [0 2 1 4 3]
round 1, starting client 1/5, id: 0
Epoch [1/1], Train Loss: 0.0049158263821288415
round 1, starting client 2/5, id: 2
Epoch [1/1], Train Loss: 0.007100741757581163
round 1, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.006489568252543936
round 1, starting client 4/5, id: 4
Epoch [1/1], Train Loss: 0.007024304106620684
round 1, starting clien

In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}.npy', rmse_lbs)

C:\Users\User\anaconda3\envs\fed\lib\site-packages\numpy\lib\npyio.py:521: VisibleDeprecationWarning:

Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.



## S=10

In [ ]:
%%time
S = 10
lstm_K5_lbs = copy.deepcopy(lstm)
outputs_lbs, targets_lbs, loss_lbs, smape_lbs, mae_lbs, rmse_lbs  = fedlbs(
    global_model = lstm_K5_lbs,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    num_clients_per_round = num_clients_per_round,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    S=S
)


starting average round 0
clients:  [0 2 1 4 3]
round 0, starting client 1/5, id: 0
Epoch [1/1], Train Loss: 0.013521558830808455
round 0, starting client 2/5, id: 2
Epoch [1/1], Train Loss: 0.015929571245310324
round 0, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.015115718315279142
round 0, starting client 4/5, id: 4
Epoch [1/1], Train Loss: 0.017690216762207032
round 0, starting client 5/5, id: 3
Epoch [1/1], Train Loss: 0.027424389977374536

sMAPE: 18.521484208039745%
calc smape: 36.712633951043045%
MAE: 8.7053557427724
RMSE: 15.327427726805748
Average Loss:  0.010892547628691615

starting average round 1
clients:  [0 2 1 4 3]
round 1, starting client 1/5, id: 0
Epoch [1/1], Train Loss: 0.004885369912028451
round 1, starting client 2/5, id: 2
Epoch [1/1], Train Loss: 0.006304962667109365
round 1, starting client 3/5, id: 1
Epoch [1/1], Train Loss: 0.00613467346067678
round 1, starting client 4/5, id: 4
Epoch [1/1], Train Loss: 0.007389413425698875
round 1, starting client 

In [ ]:
np.save(f'metrics/fedlbs_lstm_outputs_s{S}.npy', outputs_lbs)
np.save(f'metrics/fedlbs_lstm_targets_s{S}.npy', targets_lbs)
np.save(f'metrics/fedlbs_lstm_loss_s{S}.npy', loss_lbs)
np.save(f'metrics/fedlbs_lstm_smape_s{S}.npy', smape_lbs)
np.save(f'metrics/fedlbs_lstm_mae_s{S}.npy', mae_lbs)
np.save(f'metrics/fedlbs_lstm_rmse_s{S}.npy', rmse_lbs)